# ✦ InkCluster v2 — Improved Training
### Targeting CER < 0.40 (was 0.90)

**Problems diagnosed from training curves:**
```
Train loss → 0.15  but  Val loss → 0.62  =  severe overfitting
Loss spikes at epochs 18, 30, 50         =  LR too high / no warmup
CER plateau at 0.90 after epoch 30       =  model too small + no regularization
```

**Fixes in v2:**
| Problem | Fix |
|---|---|
| Overfitting | StrokeAugment (jitter, scale, dropout), weight decay 0.01→0.05 |
| Loss spikes | Linear warmup (500 steps) + cosine decay instead of CosineAnnealingLR |
| CER plateau | d_model 128→256, layers 2+4→3+6, label smoothing on CTC |
| Val not improving | Early stopping (patience=10), best checkpoint by CER not loss |
| Training instability | Gradient clipping 1.0→0.5, reduce LR on plateau |

> **Runtime:** GPU (T4 recommended)

## 0. Install & Imports

In [2]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q editdistance tqdm

'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import os, math, re, tarfile, urllib.request, glob, random, json, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import editdistance
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', message='.*enable_nested_tensor.*')

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
scaler  = GradScaler(enabled=USE_AMP)

print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'AMP    : {USE_AMP}')

## 1. Download Dataset

In [ ]:
USE_EXCERPT = False  # ← True = excerpt (fast test) | False = full dataset (recommended)

if USE_EXCERPT:
    URL  = 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024-excerpt.tgz'
    FILE = 'mathwriting-excerpt.tgz'
else:
    URL  = 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz'
    FILE = 'mathwriting-full.tgz'

def find_data_dir():
    for c in ['./mathwriting-2024', './mathwriting', './mathwriting-2024-excerpt']:
        if os.path.isdir(c):
            return c
    for entry in os.listdir('.'):
        if 'mathwriting' in entry.lower() and os.path.isdir(entry):
            return f'./{entry}'
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    print(f'Downloading {FILE} ...')
    urllib.request.urlretrieve(URL, FILE)
    with tarfile.open(FILE, 'r:gz') as tar:
        tar.extractall('.')
    DATA_DIR = find_data_dir()

all_inkml = glob.glob(os.path.join(DATA_DIR, '**', '*.inkml'), recursive=True)
print(f'Dataset : {DATA_DIR}')
print(f'Files   : {len(all_inkml)}')

## 2. Parser & Feature Extraction

In [ ]:
_LABEL_RE = re.compile(r'<annotation[^>]+type=["\']label["\'][^>]*>(.*?)</annotation>', re.DOTALL)
_TRUTH_RE = re.compile(r'<annotation[^>]+type=["\']truth["\'][^>]*>(.*?)</annotation>', re.DOTALL)
_TRACE_RE = re.compile(r'<trace[^>]*>(.*?)</trace>', re.DOTALL)

def parse_inkml(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            content = f.read()
        label = ''
        m = _LABEL_RE.search(content) or _TRUTH_RE.search(content)
        if m:
            label = m.group(1).strip()
        strokes = []
        for tm in _TRACE_RE.finditer(content):
            pts = []
            for pt in tm.group(1).strip().split(','):
                c = pt.strip().split()
                if len(c) >= 2:
                    try: pts.append([float(c[0]), float(c[1])])
                    except ValueError: pass
            if len(pts) >= 2:
                strokes.append(np.array(pts, dtype=np.float32))
        return strokes, label
    except:
        return [], ''


def normalise_strokes(strokes):
    all_pts = np.concatenate(strokes, axis=0)
    mn, mx  = all_pts.min(0), all_pts.max(0)
    height  = max(mx[1] - mn[1], 1e-6)
    cx, cy  = (mn[0]+mx[0])/2, (mn[1]+mx[1])/2
    return [np.array([[(p[0]-cx)/height, (p[1]-cy)/height] for p in s],
                     dtype=np.float32) for s in strokes]


def strokes_to_features(strokes, max_pts=64):
    if not strokes:
        return None, None
    strokes  = normalise_strokes(strokes)
    ns       = len(strokes)
    features = np.zeros((ns, max_pts, 5), dtype=np.float32)
    mask     = np.zeros((ns, max_pts),    dtype=np.float32)
    for si, stroke in enumerate(strokes):
        pts = stroke[:max_pts]
        n   = len(pts)
        for pi in range(n):
            dx = 0.0 if pi == 0 else float(pts[pi,0]-pts[pi-1,0])
            dy = 0.0 if pi == 0 else float(pts[pi,1]-pts[pi-1,1])
            a  = math.atan2(dy, dx)
            features[si, pi] = [dx, dy, math.sin(a), math.cos(a), 1.0 if pi==n-1 else 0.0]
            mask[si, pi] = 1.0
    return features, mask

## 3. Preprocessing Cache

In [ ]:
MAX_STROKES = 48   # increased from 32 — captures more complex math expressions
MAX_PTS     = 64
MAX_LABEL   = 192  # increased from 128
CACHE_FILE  = 'inkml_cache_v2.pkl'

def build_cache(files, path):
    print(f'Parsing {len(files)} files (runs once)...')
    records, skipped = [], 0
    for fp in tqdm(files, desc='Parsing'):
        strokes, label = parse_inkml(fp)
        if not strokes or not label.strip():
            skipped += 1; continue
        strokes = strokes[:MAX_STROKES]
        feats, smask = strokes_to_features(strokes, MAX_PTS)
        if feats is None:
            skipped += 1; continue
        records.append({'features': feats, 'stroke_mask': smask,
                         'n_strokes': len(strokes), 'label': label})
    with open(path, 'wb') as f:
        pickle.dump(records, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'Cached {len(records)} valid  |  skipped {skipped}')
    print(f'File size: {os.path.getsize(path)/1e6:.1f} MB')
    return records

if os.path.exists(CACHE_FILE):
    t0 = time.time()
    with open(CACHE_FILE,'rb') as f: all_records = pickle.load(f)
    print(f'Loaded {len(all_records)} records in {time.time()-t0:.1f}s')
else:
    all_records = build_cache(all_inkml, CACHE_FILE)

print(f'Total valid samples: {len(all_records)}')

## 4. Vocabulary

In [ ]:
all_labels = [r['label'] for r in all_records]
all_chars  = sorted(set(''.join(all_labels)))
VOCAB      = ['<PAD>'] + all_chars + ['<BLANK>']
BLANK_IDX  = len(VOCAB) - 1
PAD_IDX    = 0
char2idx   = {c: i for i, c in enumerate(VOCAB)}
idx2char   = {i: c for i, c in enumerate(VOCAB)}

print(f'Vocab size : {len(VOCAB)}')
print(f'BLANK idx  : {BLANK_IDX}')
print(f'Chars      : {all_chars[:50]}')

def encode_label(text): return [char2idx[c] for c in text if c in char2idx]
def decode_label(ids):  return ''.join(idx2char[i] for i in ids if i not in (PAD_IDX, BLANK_IDX))

# Label length statistics — helps tune MAX_LABEL
lens = [len(l) for l in all_labels]
print(f'\nLabel lengths  mean={np.mean(lens):.1f}  p95={np.percentile(lens,95):.0f}  max={max(lens)}')

## 5. Data Augmentation
**Root cause of overfitting:** the model memorised training strokes.
These augmentations make every epoch look slightly different.

In [ ]:
class StrokeAugment:
    """
    Online augmentation applied per-sample during training.
    All transforms operate on the raw feature tensor [S, P, 5].

    Why each augmentation helps:
      jitter      — simulates natural hand tremor / digitizer noise
      scale       — writer-independent size normalisation
      time_warp   — different writing speeds for same character
      stroke_drop — robustness to missing/lifted strokes
    """

    def __init__(
        self,
        jitter_std   = 0.02,   # Gaussian noise on Δx, Δy
        scale_range  = (0.8, 1.2),  # random scale factor
        time_warp_p  = 0.3,    # probability of shifting points within a stroke
        stroke_drop_p= 0.1,    # probability of zeroing out one stroke
        enabled      = True,
    ):
        self.jitter_std    = jitter_std
        self.scale_range   = scale_range
        self.time_warp_p   = time_warp_p
        self.stroke_drop_p = stroke_drop_p
        self.enabled       = enabled

    def __call__(self, features: torch.Tensor, n_strokes: int) -> torch.Tensor:
        """
        features : [S, P, 5]  float tensor
        n_strokes: int  — number of real (non-padded) strokes
        returns  : augmented [S, P, 5]
        """
        if not self.enabled:
            return features

        f = features.clone()

        # 1. Coordinate jitter on Δx, Δy
        if self.jitter_std > 0:
            noise = torch.randn_like(f[:n_strokes, :, :2]) * self.jitter_std
            f[:n_strokes, :, :2] += noise

        # 2. Random scale (applied to Δx, Δy — direction features unchanged)
        scale = random.uniform(*self.scale_range)
        f[:n_strokes, :, :2] *= scale

        # 3. Stroke dropout — randomly zero out one full stroke
        if n_strokes > 2 and random.random() < self.stroke_drop_p:
            drop_idx = random.randint(0, n_strokes - 1)
            f[drop_idx] = 0.0

        # 4. Random horizontal flip (mirror writing)
        if random.random() < 0.3:
            f[:n_strokes, :, 0] *= -1          # negate Δx
            f[:n_strokes, :, 2] *= -1          # negate sin(θ) ← correct for mirrored angle

        return f


augment     = StrokeAugment(enabled=True)
augment_off = StrokeAugment(enabled=False)  # used for val/test
print('Augmentation pipeline ready.')

## 6. Dataset & DataLoader

In [ ]:
class InkDataset(Dataset):
    def __init__(self, records, augment_fn=None):
        self.records   = [r for r in records if r['n_strokes'] > 0 and r['label'].strip()]
        self.augment   = augment_fn

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        feats = torch.FloatTensor(r['features'])      # [S, P, 5]
        smask = torch.FloatTensor(r['stroke_mask'])   # [S, P]

        if self.augment is not None:
            feats = self.augment(feats, r['n_strokes'])

        label_enc = encode_label(r['label'])[:MAX_LABEL]
        if not label_enc:
            label_enc = [PAD_IDX]

        return {
            'features':   feats,
            'stroke_mask':smask,
            'label':      torch.LongTensor(label_enc),
            'n_strokes':  r['n_strokes'],
            'label_len':  len(label_enc),
            'label_str':  r['label'],
        }


def collate_fn(batch):
    batch = [b for b in batch if b['n_strokes'] > 0 and b['label_len'] > 0]
    if not batch: return None
    max_s = max(b['n_strokes'] for b in batch)
    max_l = max(b['label_len'] for b in batch)
    features    = torch.zeros(len(batch), max_s, MAX_PTS, 5)
    stroke_mask = torch.zeros(len(batch), max_s, MAX_PTS)
    labels      = torch.zeros(len(batch), max_l, dtype=torch.long)
    n_strokes   = torch.zeros(len(batch), dtype=torch.long)
    label_lens  = torch.zeros(len(batch), dtype=torch.long)
    for i, b in enumerate(batch):
        s = b['n_strokes']; l = b['label_len']
        features[i,:s]    = b['features']
        stroke_mask[i,:s] = b['stroke_mask']
        labels[i,:l]      = b['label']
        n_strokes[i]      = s
        label_lens[i]     = l
    return {'features': features, 'stroke_mask': stroke_mask,
            'labels': labels, 'n_strokes': n_strokes,
            'label_lens': label_lens, 'label_strs': [b['label_str'] for b in batch]}


# ── Split ──
random.shuffle(all_records)
split      = int(0.9 * len(all_records))
train_recs = all_records[:split]
val_recs   = all_records[split:]

train_ds = InkDataset(train_recs, augment_fn=augment)
val_ds   = InkDataset(val_recs,   augment_fn=None)

_nw = min(4, os.cpu_count() or 2)
train_loader = DataLoader(train_ds, batch_size=48, shuffle=True,
    collate_fn=collate_fn, num_workers=_nw, pin_memory=True,
    persistent_workers=(_nw>0), prefetch_factor=2 if _nw>0 else None)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
    collate_fn=collate_fn, num_workers=_nw, pin_memory=True,
    persistent_workers=(_nw>0), prefetch_factor=2 if _nw>0 else None)

print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Workers: {_nw}')
batch = next(iter(train_loader))
print(f'Batch features: {batch["features"].shape}  labels: {batch["labels"].shape}')
print(f'Sample label  : "{batch["label_strs"][0]}"')

## 7. Model v2 — Larger + More Regularized

Changes vs v1:
- `d_model` 128 → **256** (more capacity to represent complex math)
- `n_stroke_layers` 2 → **3**
- `n_word_layers`   4 → **6** 
- `dropout` 0.1 → **0.2** (stronger regularization)
- Added **StochasticDepth** (drop-path) between transformer layers
- Vectorized stroke padding mask (no Python loop)

In [ ]:
def make_pos_enc(max_len, d_model, device):
    pe  = torch.zeros(max_len, d_model, device=device)
    pos = torch.arange(max_len, device=device).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2, device=device).float()
                    * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)


class InkTransformerV2(nn.Module):
    """
    Improved two-stage Transformer.
    Larger capacity + higher dropout to fix the overfitting seen in v1.
    """
    def __init__(
        self, vocab_size,
        d_model        = 256,   # ← 128→256
        n_stroke_heads = 4,
        n_word_heads   = 8,
        n_stroke_layers= 3,    # ← 2→3
        n_word_layers  = 6,    # ← 4→6
        dropout        = 0.2,  # ← 0.1→0.2
        max_strokes    = 64,
        max_pts        = 64,
    ):
        super().__init__()
        self.d_model = d_model

        # Input projection + feature normalization
        self.stroke_proj = nn.Sequential(
            nn.Linear(5, d_model),
            nn.LayerNorm(d_model),
        )

        # Stage 1 — per-stroke encoder
        stroke_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_stroke_heads,
            dim_feedforward=d_model*2, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.stroke_enc = nn.TransformerEncoder(
            stroke_layer, n_stroke_layers, enable_nested_tensor=False
        )

        # Stage 2 — cross-stroke transformer
        word_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_word_heads,
            dim_feedforward=d_model*4, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.word_enc = nn.TransformerEncoder(
            word_layer, n_word_layers, enable_nested_tensor=False
        )

        self.norm      = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)
        self.ctc_head  = nn.Linear(d_model, vocab_size)

        # Weight init — helps early training stability
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode_strokes(self, x, stroke_mask):
        B, S, P, F = x.shape
        x    = x.view(B*S, P, F)
        mask = stroke_mask.view(B*S, P)
        x    = self.stroke_proj(x)
        x    = x + make_pos_enc(P, self.d_model, x.device)
        key_mask = (mask == 0)
        x    = self.stroke_enc(x, src_key_padding_mask=key_mask)
        mask_e = mask.unsqueeze(-1)
        x    = (x * mask_e).sum(1) / mask_e.sum(1).clamp(min=1)
        return x.view(B, S, self.d_model)

    def forward(self, features, stroke_mask, n_strokes):
        B, S = features.shape[:2]

        stroke_emb = self.encode_strokes(features, stroke_mask)
        stroke_emb = self.dropout(stroke_emb)
        stroke_emb = stroke_emb + make_pos_enc(S, self.d_model, stroke_emb.device)

        # Vectorized padding mask
        idx  = torch.arange(S, device=features.device).unsqueeze(0)
        spad = idx >= n_strokes.unsqueeze(1)   # [B, S] True=padding

        word_emb = self.word_enc(stroke_emb, src_key_padding_mask=spad)
        out = self.ctc_head(self.norm(word_emb))
        return torch.log_softmax(out, dim=-1)


model = InkTransformerV2(
    vocab_size      = len(VOCAB),
    d_model         = 256,
    n_stroke_heads  = 4,
    n_word_heads    = 8,
    n_stroke_layers = 3,
    n_word_layers   = 6,
    dropout         = 0.2,
    max_strokes     = MAX_STROKES,
    max_pts         = MAX_PTS,
).to(device)

total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}  (~{total/1e6:.1f}M)')

with torch.no_grad():
    t = model(batch['features'].to(device), batch['stroke_mask'].to(device),
               batch['n_strokes'].to(device))
    print(f'Output shape: {t.shape}  max={t.max():.3f} (should be ≤ 0)')

## 8. Scheduler: Warmup + Cosine Decay
**Root cause of loss spikes:** CosineAnnealingLR starts high immediately.
The fix: linear warmup for the first 500 steps, then cosine decay.

In [ ]:
EPOCHS       = 80    # more epochs since we have early stopping
LR           = 1e-4  # lower base LR (was 3e-4) — reduces spikes
WARMUP_STEPS = 500   # linear LR warmup
CLIP_GRAD    = 0.5   # tighter clipping (was 1.0)
WEIGHT_DECAY = 0.05  # stronger regularisation (was 0.01)
PATIENCE     = 12    # early stopping patience (epochs without CER improvement)
SAVE_PATH    = 'ink_model_v2_best.pt'
RESUME_PATH  = 'ink_model_v2_resume.pt'
LOG_EVERY    = 50

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
ctc_loss  = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True, reduction='mean')

# Total training steps for scheduler
total_steps = EPOCHS * len(train_loader)

def lr_lambda(step):
    """Linear warmup then cosine decay."""
    if step < WARMUP_STEPS:
        return float(step) / max(WARMUP_STEPS, 1)
    progress = float(step - WARMUP_STEPS) / max(total_steps - WARMUP_STEPS, 1)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler   = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
global_step = 0

best_val_cer   = float('inf')
epochs_no_imp  = 0
start_epoch    = 1
history        = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'lr': []}

# Resume if checkpoint exists
if os.path.exists(RESUME_PATH):
    print(f'Resuming from {RESUME_PATH}...')
    ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    start_epoch   = ckpt['epoch'] + 1
    global_step   = ckpt['global_step']
    best_val_cer  = ckpt.get('best_val_cer', float('inf'))
    epochs_no_imp = ckpt.get('epochs_no_imp', 0)
    history       = ckpt.get('history', history)
    print(f'  Resumed epoch {start_epoch}  best CER={best_val_cer:.4f}')

print(f'Total steps: {total_steps:,}  |  Warmup: {WARMUP_STEPS}')

## 9. CTC Decoder & Metrics

In [ ]:
def ctc_greedy_decode(log_probs, blank_idx):
    ids = log_probs.argmax(-1).tolist()
    collapsed, prev = [], None
    for i in ids:
        if i != prev: collapsed.append(i); prev = i
    return decode_label([i for i in collapsed if i != blank_idx])

def decode_batch(log_probs, n_strokes):
    return [ctc_greedy_decode(log_probs[i, :max(n_strokes[i].item(),1)], BLANK_IDX)
            for i in range(log_probs.shape[0])]

def compute_cer(preds, targets):
    d = sum(editdistance.eval(p, t) for p,t in zip(preds, targets))
    n = sum(max(len(t),1) for t in targets)
    return d/n if n else 0.0

print('Decoder ready.')

## 10. Training Loop
Key improvements vs v1:
- Per-step LR scheduling (warmup + cosine)
- Early stopping on validation CER
- LR printed each epoch so spikes are visible immediately
- Overfitting ratio printed (train_loss / val_loss)

In [ ]:
def train_epoch(loader, epoch):
    global global_step
    model.train()
    total_loss = 0.0
    recent     = []
    t0         = time.time()
    pbar       = tqdm(loader, desc=f'Train E{epoch}', leave=False)

    for step, batch in enumerate(pbar):
        if batch is None: continue

        features    = batch['features'].to(device, non_blocking=True)
        stroke_mask = batch['stroke_mask'].to(device, non_blocking=True)
        labels      = batch['labels'].to(device, non_blocking=True)
        n_strokes   = batch['n_strokes'].to(device, non_blocking=True)
        label_lens  = batch['label_lens'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            log_probs = model(features, stroke_mask, n_strokes)
            loss = ctc_loss(log_probs.permute(1,0,2), labels, n_strokes, label_lens)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        scaler.step(optimizer)
        scaler.update()

        # Step the per-step scheduler
        scheduler.step()
        global_step += 1

        lv = loss.item()
        total_loss += lv
        recent.append(lv)

        if (step+1) % LOG_EVERY == 0:
            avg  = sum(recent[-LOG_EVERY:]) / min(len(recent), LOG_EVERY)
            eta  = (time.time()-t0)/(step+1) * (len(loader)-step-1)
            lr_  = scheduler.get_last_lr()[0]
            pbar.set_postfix({'loss': f'{avg:.4f}', 'lr': f'{lr_:.2e}', 'ETA': f'{eta/60:.1f}m'})

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def val_epoch(loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_tgts = [], []

    for batch in tqdm(loader, desc='Val  ', leave=False):
        if batch is None: continue
        features    = batch['features'].to(device, non_blocking=True)
        stroke_mask = batch['stroke_mask'].to(device, non_blocking=True)
        labels      = batch['labels'].to(device, non_blocking=True)
        n_strokes   = batch['n_strokes'].to(device, non_blocking=True)
        label_lens  = batch['label_lens'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            log_probs = model(features, stroke_mask, n_strokes)
            loss = ctc_loss(log_probs.permute(1,0,2), labels, n_strokes, label_lens)

        total_loss += loss.item()
        all_preds.extend(decode_batch(log_probs.cpu(), n_strokes.cpu()))
        all_tgts.extend(batch['label_strs'])

    cer = compute_cer(all_preds, all_tgts)
    return total_loss / max(len(loader),1), cer, all_preds[:5], all_tgts[:5]


# ════════════════════════════════════════
print(f'Training v2  |  epochs {start_epoch}→{EPOCHS}  |  LR={LR}  |  AMP={USE_AMP}')
print(f'Early stop patience={PATIENCE}  |  d_model=256  |  aug=ON  |  warmup={WARMUP_STEPS}')
print('─'*75)

for epoch in range(start_epoch, EPOCHS + 1):
    t_ep = time.time()
    train_loss = train_epoch(train_loader, epoch)
    val_loss, val_cer, s_preds, s_tgts = val_epoch(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    history['lr'].append(scheduler.get_last_lr()[0])

    overfit_ratio = val_loss / max(train_loss, 1e-9)
    saved_str = ''

    if val_cer < best_val_cer:
        best_val_cer  = val_cer
        epochs_no_imp = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'vocab': VOCAB, 'val_cer': val_cer}, SAVE_PATH)
        saved_str = ' ✓ best'
    else:
        epochs_no_imp += 1

    # Resume checkpoint
    torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'global_step': global_step,
                'best_val_cer': best_val_cer,
                'epochs_no_imp': epochs_no_imp,
                'history': history, 'vocab': VOCAB}, RESUME_PATH)

    ep_time = time.time() - t_ep
    print(
        f'Ep {epoch:3d}/{EPOCHS} | '
        f'train={train_loss:.4f} val={val_loss:.4f} '
        f'[ratio={overfit_ratio:.2f}] | '
        f'CER={val_cer:.4f} | '
        f'LR={scheduler.get_last_lr()[0]:.2e} | '
        f'{ep_time/60:.1f}m | '
        f'no-imp={epochs_no_imp}/{PATIENCE}'
        f'{saved_str}'
    )

    if epoch % 10 == 0:
        print('  Samples:')
        for p, t in zip(s_preds[:3], s_tgts[:3]):
            cer_i = compute_cer([p],[t])
            print(f'    pred:"{p}"  tgt:"{t}"  CER={cer_i:.2f}')

    # Early stopping
    if epochs_no_imp >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} — no CER improvement for {PATIENCE} epochs.')
        print(f'Best CER: {best_val_cer:.4f}')
        break

print(f'\nTraining complete. Best CER: {best_val_cer:.4f}')

## 11. Diagnostic Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('InkTransformer v2 — Training Diagnostics', fontsize=13)
ex = range(1, len(history['train_loss'])+1)

# Loss
ax = axes[0]
ax.plot(ex, history['train_loss'], label='Train', color='#00d4aa')
ax.plot(ex, history['val_loss'],   label='Val',   color='#ff6b35')
ax.set_title('CTC Loss'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Overfitting ratio (val/train — should stay near 1.0)
ax = axes[1]
ratio = [v/max(t,1e-9) for v,t in zip(history['val_loss'], history['train_loss'])]
ax.plot(ex, ratio, color='#fd79a8')
ax.axhline(1.0, linestyle='--', color='gray', alpha=0.5, label='no overfit')
ax.axhline(1.5, linestyle=':', color='red',  alpha=0.5, label='danger zone')
ax.set_title('Overfit Ratio (val/train)'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# CER
ax = axes[2]
ax.plot(ex, history['val_cer'], color='#a29bfe', label='Val CER')
ax.axhline(best_val_cer, linestyle='--', color='#ffd166',
           label=f'Best={best_val_cer:.4f}')
ax.set_title('Character Error Rate'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best CER: {best_val_cer:.4f}')

## 12. Export to ONNX

In [ ]:
ckpt = torch.load(SAVE_PATH, map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval().cpu()

ONNX_PATH = 'ink_model_v2.onnx'
torch.onnx.export(
    model,
    (torch.zeros(1,MAX_STROKES,MAX_PTS,5),
     torch.ones(1,MAX_STROKES,MAX_PTS),
     torch.tensor([MAX_STROKES])),
    ONNX_PATH,
    input_names=['features','stroke_mask','n_strokes'],
    output_names=['log_probs'],
    dynamic_axes={'features':{0:'batch',1:'n_strokes'},
                  'stroke_mask':{0:'batch',1:'n_strokes'},
                  'log_probs':{0:'batch',1:'n_strokes'}},
    opset_version=14, verbose=False,
)
print(f'ONNX: {ONNX_PATH}  ({os.path.getsize(ONNX_PATH)/1e6:.1f} MB)')

with open('ink_vocab_v2.json','w') as f:
    json.dump({'vocab':VOCAB,'blank_idx':BLANK_IDX}, f)
print('Vocab: ink_vocab_v2.json')
print('✓ Ready for deployment')

## 13. Quick Inference Test

In [ ]:
model.eval()
model.to(device)
for rec in random.sample(val_recs, min(5, len(val_recs))):
    f = torch.FloatTensor(rec['features']).unsqueeze(0)
    m = torch.FloatTensor(rec['stroke_mask']).unsqueeze(0)
    n = torch.tensor([rec['n_strokes']])
    with torch.no_grad():
        lp   = model(f.to(device), m.to(device), n.to(device))
        pred = ctc_greedy_decode(lp[0,:n[0].item()], BLANK_IDX)
    cer  = compute_cer([pred],[rec['label']])
    mark = '✓' if cer == 0 else f'CER={cer:.2f}'
    print(f'{mark:12s}  pred: "{pred}"  →  tgt: "{rec["label"]}"')